# Text Summarization menggunakan TF-IDF
## Artikel Pertamina - Extractive Summarization (Teks Bahasa Indonesia)

**Tujuan:** Menghasilkan ringkasan ekstraktif dari artikel Pertamina Integrated Supply Chain (ISC) menggunakan metodologi penskoran TF-IDF dengan pemrosesan bahasa Indonesia.

**Metode:** 
- Mengekstrak kalimat berdasarkan skor pentingnya TF-IDF
- Menggunakan penghapus stopword Sastrawi yang dioptimalkan untuk Bahasa Indonesia
- Menerapkan vektorisasi TF-IDF pada teks Bahasa Indonesia
- Menghitung threshold sebagai rata-rata dari semua skor kalimat
- Memilih kalimat dengan skor >= threshold

**Output:** Ringkasan ringkas yang mempertahankan urutan kalimat asli dan bahasa Indonesia

## Langkah 1: Import Library yang Diperlukan

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Initialize Indonesian StopWordRemover (Sastrawi)
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

print("✓ Semua library berhasil diimpor")
print("✓ Penghapus StopWord Bahasa Indonesia (Sastrawi) telah diinisialisasi")

## Langkah 2: Tentukan Dokumen (Artikel Pertamina ISC - Teks Bahasa Indonesia)

In [ ]:
sentence = """
JAKARTA - Melalui kebijakan yang transparan serta memangkas mata rantai dalam proses pengadaan minyak mentah serta produk BBM yang sebelumnya dijalankan PT Pertamina Energy Trading Ltd (Petral), Integrated Supply Chain (ISC) Pertamina berhasil mencatatkan efisiensi bagi perusahaan senilai US$ 208,1 juta atau setara Rp 2,87 triliun (kurs Rp 13.800) sepanjang 2015.

Capaian efisiensi tersebut diperoleh melalui lima program terobosan ISC yang disebut dengan fase ISC 1.0.

Vice President Corporate Communication PT Pertamina (Persero) Wianda Pusponegoro mengatakan lima program fase ISC 1.0 itu adalah memotong perantara dari rantai suplai, peningkatan pemanfaatan dan fleksibilitas dari armada laut Pertamina.

Terobosan lainnya adalah dengan pemberian kesempatan yang sama dan adil untuk semua peserta pengadaan.

\"ISC 1.0 juga menerapkan terobosan lain berupa penerapan proses evaluasi penawaran yang transparan dan mengurangi biaya dengan menerapkan pembayaran telegraphic transfer (TT),\" ujar Wianda.

Wianda menjelaskan keberadaan ISC sangat penting untuk membuka transparansi seluas-luasnya supaya banyak mitra terpilih yang ikut serta.

Dengan demikian, ada perubahan yang signifikan berupa penghematan.

\"Kami bisa memangkas rantai suplai pengadaan impor, dimana untuk minyak maupun produk, Indonesia masih. Ini yang kami kejar terus,\" katanya.

Transformasi ISC adalah bagian dari upaya meningkatkan efisiensi dan memperkuat transparansi pengadaan minyak mentah dan produk minyak yang selalu menjadi perhatian publik.

Perkuatan ISC dengan mengembalikan fungsi ISC dan sekaligus meningkatkan kewenangannya.

Menurut Wianda, efisiensi dari sisi pengadaan minyak mentah dan bahan bakar minyak (BBM) yang dilakukan ISC salah satunya dilakukan dengan mengevaluasi ulang kontrak-kontrak pembelian sebelumnya.

\"ISC akan melakukan upaya terbaik untuk mendapatkan harga yang paling optimal bagi Pertamina,\" ujarnya.

Pertamina, menurut Wianda, mengundang daftar mitra usaha terseleksi (DMUT) untuk terlibat dalam pengadaan minyak mentah dan produk BBM secara terbuka dan transparan.

Penetapan DMUT juga cukup ketat karena harus memenuhi sejumlah kualifikasi tertentu seperti detail bisnis perusahaan, detail laporan keuangan, detail bank, dan lain-lain.

\"Melalui ISC, peserta tender lebih variatif, harga lebih kompetitif, dan posisi tawar semakin tinggi.

Informasi tender kami buka melalui website Pertamina yang semua orang dapat mengaksesnya,\" jelas dia.

Wianda mengatakan dengan transparansi menjadikan ISC menjalankan fungsi pengadaan lebih baik, kompetitif, dan menghilangkan potensi conflict of interest.

\"Pola mekanisme tender yang dilakukan melalui email dan metode evaluasi penawaran ketat dan prudent menjadikan proses pengadaan minyak dan produk Pertamina lebih akuntabel dan kredibel.\"

Wianda menegaskan efisiensi dalam pengadaan juga dilakukan dengan mengoptimalkan penggunaan kapal-kapal yang dikelola oleh Pertamina, baik untuk mengangkut BBM, minyak mentah, dan elpiji impor dari titik penjualan ke dalam negeri.

Keberhasilan Pertamina lainnya dalam pengadaan adalah adanya renegosiasi kontrak dengan Saudi Aramco, yang memiliki kontrak evergreen dengan Pertamina sekitar 120.000 barel per hari.

Sejak Juni 2015, Saudi Aramco bersedia untuk tidak lagi mewajibkan Pertamina menerbitkan letter of credit (L/C).

\"Ini adalah bentuk kepercayaan dari pemasok minyak mentah kepada Pertamina dalam kaitannya dengan jaminan pembayaran.

Berapa sen pun yang dapat kami hemat, Pertamina akan lakukan upaya terbaik untuk meraihnya, tentu saja sesuai dengan kaidah-kaidah dan best practices yang ada,\" tutup Wianda.
"""

print("✓ Dokumen berhasil dimuat")
print(f"Total karakter: {len(sentence)}")
print(f"Bahasa: Bahasa Indonesia")

## Langkah 3: Pra-pemrosesan - Tokenisasi Kalimat

In [ ]:
print("="*80)
print("LANGKAH 1: TOKENISASI KALIMAT (TEKS BAHASA INDONESIA)")
print("="*80)

sent_token = sent_tokenize(sentence)
print(f"\nTotal kalimat yang diekstrak: {len(sent_token)}\n")

for idx, sent in enumerate(sent_token, 1):
    print(f"{idx:2d}. {sent[:80]}...")

## Langkah 4: Penghapusan Stopword Bahasa Indonesia & Vektorisasi TF-IDF

In [ ]:
print("\n" + "="*80)
print("LANGKAH 2: PENGHAPUSAN STOPWORD BAHASA INDONESIA (SASTRAWI) & VEKTORISASI TF-IDF")
print("="*80)

# Menghapus stopword Bahasa Indonesia dari setiap kalimat menggunakan Sastrawi
cleaned_sentences = [stopword_remover.remove(sent) for sent in sent_token]

print("\nContoh: Kalimat Asli vs Kalimat Dibersihkan (3 Pertama):")
print("-" * 80)
for i in range(min(3, len(sent_token))):
    print(f"\n[{i+1}] Asli: {sent_token[i][:65]}...")
    print(f"    Dibersihkan: {cleaned_sentences[i][:65]}...")

print("\n" + "-"*80)

# Menerapkan TF-IDF Vectorizer pada kalimat yang dibersihkan
vectorizer = TfidfVectorizer()
features = vectorizer.fit_transform(cleaned_sentences)

print(f"\nBentuk Matriks Fitur: {features.shape}")
print(f"  - Jumlah kalimat: {features.shape[0]}")
print(f"  - Jumlah kata unik (fitur): {features.shape[1]}")

feature_names = vectorizer.get_feature_names_out()
print(f"\n20 Kata Penting Teratas dalam Kosakata:")

# Dapatkan kata teratas menurut rata-rata TF-IDF
avg_tfidf = features.mean(axis=0).A1
top_indices = avg_tfidf.argsort()[-20:][::-1]
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. {feature_names[idx]:20s} - {avg_tfidf[idx]:.4f}")

## Langkah 5: Hitung Skor Kalimat

In [ ]:
print("\n" + "="*80)
print("LANGKAH 3: HITUNG SKOR TF-IDF RATA-RATA UNTUK SETIAP KALIMAT")
print("="*80)

sent_scores = []
print("\nSkor Kalimat:")
print("-" * 80)

for idx, tfidf_vector in enumerate(features, 1):
    sent_score = tfidf_vector.sum()
    sent_length = len(tfidf_vector.data)
    avg_score = sent_score / sent_length if sent_length > 0 else 0
    sent_scores.append(avg_score)
    print(f"Kalimat {idx:2d}: Skor = {avg_score:.4f} (jumlah={sent_score:.4f}, kata={sent_length})")

# Hitung threshold (batas ambang)
threshold = sum(sent_scores) / len(sent_scores)
print(f"\n{'─'*80}")
print(f"BATAS AMBANG (Skor Rata-rata): {threshold:.4f}")
print(f"{'─'*80}")

## Langkah 6: Visualisasi Skor Kalimat

In [ ]:
print("\nMembuatkan visualisasi...\n")

plt.figure(figsize=(15, 6))
colors = ['#2ecc71' if score >= threshold else '#e74c3c' for score in sent_scores]
bars = plt.bar(range(1, len(sent_scores) + 1), sent_scores, color=colors, edgecolor='black', linewidth=1.2)

# Tambahkan garis threshold (batas ambang)
plt.axhline(y=threshold, color='#3498db', linestyle='--', linewidth=2.5, label=f'Batas Ambang: {threshold:.4f}')

plt.xlabel("Nomor Kalimat", fontsize=12, fontweight='bold')
plt.ylabel("Skor TF-IDF Rata-rata", fontsize=12, fontweight='bold')
plt.title("Skor TF-IDF Rata-rata per Kalimat\nArtikel Pertamina ISC (Teks Bahasa Indonesia)", fontsize=14, fontweight='bold')
plt.xticks(range(1, len(sent_scores) + 1), fontsize=10)
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Add color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='Dipilih (≥ batas ambang)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Tidak dipilih (< batas ambang)')
]
plt.legend(handles=legend_elements + [plt.Line2D([0], [0], color='#3498db', linewidth=2.5, linestyle='--')], 
           labels=['Dipilih (≥ batas ambang)', 'Tidak dipilih (< batas ambang)', f'Batas Ambang: {threshold:.4f}'],
           fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

print("✓ Visualisasi selesai")

## Langkah 7: Hasilkan Ringkasan Ekstraktif

In [ ]:
print("\n" + "="*80)
print("LANGKAH 4: HASILKAN RINGKASAN EKSTRAKTIF (TEKS BAHASA INDONESIA)")
print("="*80)

final_summ = ""
summary_indices = []

print(f"\n**Kalimat yang Dipilih untuk Ringkasan (Skor >= {threshold:.4f}):**\n")
print("-" * 80)

for idx, score in enumerate(sent_scores, 1):
    if score >= threshold:
        summary_indices.append(idx)
        final_summ = final_summ + " " + sent_token[idx-1]
        print(f"✓ [{idx:2d}] {sent_token[idx-1][:70]}...")
    else:
        print(f"✗ [{idx:2d}] {sent_token[idx-1][:70]}...")

print("\n" + "-"*80)

## Langkah 8: Tampilkan Ringkasan Akhir & Statistik

In [ ]:
print("\n" + "="*80)
print("RINGKASAN AKHIR - ARTIKEL PERTAMINA ISC (TEKS BAHASA INDONESIA)")
print("="*80)

total_original_words = sum(len(s.split()) for s in sent_token)
total_summary_words = len(final_summ.split())
compression_ratio = (total_summary_words / total_original_words * 100)

print(f"\n📊 STATISTIK:")
print(f"  • Teks asli: {total_original_words} kata")
print(f"  • Ringkasan: {total_summary_words} kata")
print(f"  • Rasio kompresi: {compression_ratio:.1f}%")
print(f"  • Kalimat yang dipilih: {len(summary_indices)}/{len(sent_token)}")
print(f"  • Pengurangan kalimat: {((1 - len(summary_indices)/len(sent_token))*100):.1f}%")

print("\n" + "-"*80)
print("\n📝 TEKS RINGKASAN (DALAM BAHASA INDONESIA):\n")
print(final_summ.strip())
print("\n" + "="*80)

## Langkah 9: Tabel Analisis Detail

In [ ]:
print("\n📋 ANALISIS KALIMAT DETAIL (ARTIKEL PERTAMINA - BAHASA INDONESIA):")
print()

summary_df = pd.DataFrame({
    'No. Kalimat': range(1, len(sent_token) + 1),
    'Skor': [f'{s:.4f}' for s in sent_scores],
    'Dipilih': ['✓' if i+1 in summary_indices else '✗' for i in range(len(sent_token))],
    'Status': ['DIMASUKKAN' if i+1 in summary_indices else 'DIKECUALIKAN' for i in range(len(sent_token))],
    'Pratinjau Teks': [s[:50] + '...' if len(s) > 50 else s for s in sent_token]
})

print(summary_df.to_string(index=False))
print("\n" + "="*80)

## Penjelasan Algoritma Ringkasan (Pemrosesan Teks Bahasa Indonesia)

### Langkah-Langkah Prapemrosesan untuk Teks Bahasa Indonesia:

1. **Tokenisasi Kalimat**: Teks dipisahkan menjadi kalimat-kalimat individu menggunakan tokenizer NLTK punkt
2. **Penghapusan Stopword Bahasa Indonesia (Sastrawi)**: Menghapus kata-kata umum Bahasa Indonesia (yang, dari, untuk, dll)
3. **Kalimat yang Dibersihkan**: Hanya kata-kata bermakna yang tersisa untuk analisis TF-IDF

### Proses Perhitungan TF-IDF:

1. **TF (Term Frequency)**: Seberapa sering sebuah kata muncul di setiap kalimat yang dibersihkan
2. **IDF (Inverse Document Frequency)**: Seberapa unik/penting sebuah kata di semua kalimat
3. **TF-IDF = TF × IDF**: Menggabungkan kedua ukuran untuk mengidentifikasi kata-kata penting
4. **Penskoran Kalimat**: Skor TF-IDF rata-rata dari semua kata bermakna di setiap kalimat
5. **Batas Ambang**: Rata-rata dari semua skor kalimat
6. **Pemilihan**: Kalimat dengan skor ≥ batas ambang dipilih untuk ringkasan
7. **Ringkasan**: Kalimat terpilih disusun dalam urutan asli

### Fitur Kunci untuk Teks Bahasa Indonesia:

- ✓ **Penghapus Stopword Sastrawi**: Dioptimalkan khusus untuk bahasa Indonesia
- ✓ **Batang hijau** = Kalimat cukup penting untuk dimasukkan dalam ringkasan
- ✗ **Batang merah** = Kalimat dengan informasi kurang penting
- **Garis putus-putus biru** = Batas keputusan (batas ambang)

### Keuntungan Metode Ini:

- ✓ Cepat dan efisien untuk artikel berita/korporat Bahasa Indonesia
- ✓ Tidak memerlukan data pelatihan - pembelajaran tanpa pengawasan
- ✓ Mempertahankan urutan kalimat asli dan konteks
- ✓ Bekerja dengan baik untuk artikel bisnis/korporat/keuangan
- ✓ Hasil transparan dan dapat diinterpretasi
- ✗ Tidak dapat menulis ulang atau mengompresi kalimat individual
- ✗ "Ekstraktif" saja (memilih kalimat yang ada, tidak membuat teks baru sama sekali)

### Konteks Artikel:

**Laporan Efisiensi Integrated Supply Chain (ISC) Pertamina 2015**
- **Topik**: Peningkatan efisiensi rantai suplai Pertamina
- **Pencapaian Utama**: Penghematan biaya sebesar US$208,1 juta (Rp 2,87 triliun)
- **Program**: ISC 1.0 - Lima program terobosan
- **Tahun**: 2015
- **Bahasa**: Bahasa Indonesia
- **Tipe Artikel**: Berita Korporat/Bisnis